# Imports

In [23]:
import re
import pickle

from catboost import CatBoostClassifier
import pandas as pd

# Basic lists

In [24]:
# Списки объедениящие группы признаков

# rn - уникальный признак

# Бинаризированные
pre_features = [
    'pre_since_opened',
    'pre_since_confirmed',
    'pre_pterm',
    'pre_fterm',
    'pre_till_pclose',
    'pre_till_fclose',
    'pre_loans_credit_limit',
    'pre_loans_next_pay_summ',
    'pre_loans_outstanding',
    'pre_loans_max_overdue_sum',
    'pre_loans_credit_cost_rate',
    'pre_loans5',
    'pre_loans530',
    'pre_loans3060',
    'pre_loans6090',
    'pre_loans90',
    'pre_util',
    'pre_over2limit',
    'pre_maxover2limit'
]

# Закодированные
enc_features = [
    'enc_loans_account_holder_type',
    'enc_loans_credit_status',
    'enc_loans_credit_type',
    'enc_loans_account_cur'
]

# Статусы ежемесячных платежей
enc_paym_features = [
    'enc_paym_0',
    'enc_paym_1',
    'enc_paym_2',
    'enc_paym_3',
    'enc_paym_4',
    'enc_paym_5',
    'enc_paym_6',
    'enc_paym_7',
    'enc_paym_8',
    'enc_paym_9',
    'enc_paym_10',
    'enc_paym_11',
    'enc_paym_12',
    'enc_paym_13',
    'enc_paym_14',
    'enc_paym_15',
    'enc_paym_16',
    'enc_paym_17',
    'enc_paym_18',
    'enc_paym_19',
    'enc_paym_20',
    'enc_paym_21',
    'enc_paym_22',
    'enc_paym_23',
    'enc_paym_24'
]

#  Флаги
flag_features = [
    'is_zero_loans5',
    'is_zero_loans530',
    'is_zero_loans3060',
    'is_zero_loans6090',
    'is_zero_loans90',
    'is_zero_util',
    'is_zero_over2limit',
    'is_zero_maxover2limit',
    'pclose_flag',
    'fclose_flag'
]

In [25]:
# Исходный датасет
df_source = pd.read_csv('../data/processed/source_data_train_1.csv')
df_source.shape

(20931476, 61)

In [26]:
"""
Датасет сгенерированных признаков. 
Результат feature engineering и обрезки лишних фичей.
"""
df_result = pd.read_csv('../data/processed/cut_corr_imp_train.csv')
df_result.shape

(2400000, 61)

In [27]:
df_source_columns = df_source.columns.tolist()
df_source_columns[:10]

['id',
 'rn',
 'pre_since_opened',
 'pre_since_confirmed',
 'pre_pterm',
 'pre_fterm',
 'pre_till_pclose',
 'pre_till_fclose',
 'pre_loans_credit_limit',
 'pre_loans_next_pay_summ']

In [28]:
df_result_columns = df_result.columns.tolist()
df_result_columns[:10]

['id',
 'flag',
 'is_zero_sum_prop_1',
 'enc_paym_avg_0_1_this_year_diff',
 'pre_util_prop_3',
 'enc_loans_credit_type_prop_0',
 'pre_till_pclose_prop_10',
 'pre_util_prop_6',
 'pre_loans_outstanding_prop_1',
 'pre_util_mean_freq']

# List of features to download from the original dataset

In [29]:
"""
Формируем список колонок из df_source_columns,
которые НЕ встречаются ни в одном названии из df_result_columns как подстрока.
"""
drop_list = []
for col_source in df_source_columns:
    found = False
    for col_result in df_result_columns:
        if col_source in col_result:
            found = True
            break
    if not found:
        drop_list.append(col_source)

print(len(drop_list))
drop_list

30


['pre_loans_total_overdue',
 'pre_loans3060',
 'pre_loans6090',
 'pre_loans90',
 'is_zero_loans3060',
 'is_zero_loans6090',
 'is_zero_loans90',
 'pre_maxover2limit',
 'is_zero_util',
 'is_zero_maxover2limit',
 'enc_paym_3',
 'enc_paym_4',
 'enc_paym_5',
 'enc_paym_6',
 'enc_paym_7',
 'enc_paym_11',
 'enc_paym_12',
 'enc_paym_13',
 'enc_paym_14',
 'enc_paym_15',
 'enc_paym_16',
 'enc_paym_17',
 'enc_paym_18',
 'enc_paym_19',
 'enc_paym_20',
 'enc_paym_21',
 'enc_paym_22',
 'enc_paym_23',
 'pclose_flag',
 'fclose_flag']

In [30]:
"""
Выберем из списка колонок исходного датасета те которые не
встречаются в предыдущем списке.
"""
pre_features = [x for x in df_source_columns if x not in drop_list]

print(len(pre_features))
pre_features[:10]

31


['id',
 'rn',
 'pre_since_opened',
 'pre_since_confirmed',
 'pre_pterm',
 'pre_fterm',
 'pre_till_pclose',
 'pre_till_fclose',
 'pre_loans_credit_limit',
 'pre_loans_next_pay_summ']

In [31]:
"""
Добавим недостающие признаки из групп flag_features и enc_paym _features, 
для правильной работы функций обрабатывающих эти группы. 
"""
features_list= [
    'is_zero_loans3060',
    'is_zero_loans6090',
    'is_zero_loans90',
    'enc_paym_3',
    'enc_paym_4',
    'enc_paym_5',
    'enc_paym_6',
    'enc_paym_7',
    'enc_paym_11',
    'enc_paym_12',
    'enc_paym_13',
    'enc_paym_14',
    'enc_paym_15',
    'enc_paym_16',
    'enc_paym_17',
    'enc_paym_18',
    'enc_paym_19',
    'enc_paym_20',
    'enc_paym_21',
    'enc_paym_22',
    'enc_paym_23'
]

# Список признаков для скачивания из исходного датасета
pre_features = pre_features + features_list

print(len(pre_features))
pre_features

52


['id',
 'rn',
 'pre_since_opened',
 'pre_since_confirmed',
 'pre_pterm',
 'pre_fterm',
 'pre_till_pclose',
 'pre_till_fclose',
 'pre_loans_credit_limit',
 'pre_loans_next_pay_summ',
 'pre_loans_outstanding',
 'pre_loans_max_overdue_sum',
 'pre_loans_credit_cost_rate',
 'pre_loans5',
 'pre_loans530',
 'is_zero_loans5',
 'is_zero_loans530',
 'pre_util',
 'pre_over2limit',
 'is_zero_over2limit',
 'enc_paym_0',
 'enc_paym_1',
 'enc_paym_2',
 'enc_paym_8',
 'enc_paym_9',
 'enc_paym_10',
 'enc_paym_24',
 'enc_loans_account_holder_type',
 'enc_loans_credit_status',
 'enc_loans_credit_type',
 'enc_loans_account_cur',
 'is_zero_loans3060',
 'is_zero_loans6090',
 'is_zero_loans90',
 'enc_paym_3',
 'enc_paym_4',
 'enc_paym_5',
 'enc_paym_6',
 'enc_paym_7',
 'enc_paym_11',
 'enc_paym_12',
 'enc_paym_13',
 'enc_paym_14',
 'enc_paym_15',
 'enc_paym_16',
 'enc_paym_17',
 'enc_paym_18',
 'enc_paym_19',
 'enc_paym_20',
 'enc_paym_21',
 'enc_paym_22',
 'enc_paym_23']

# Definite_value_proportion_features_pipeline funtion list

In [32]:
"""
Создадим список пропорциональных фичей в итоговом датасете
созданных функцией create_definite_value_proportion_features.
"""
prop_features_result_list = [col for col in df_result_columns if 'prop_' in col]

print(len(prop_features_result_list))
prop_features_result_list

38


['is_zero_sum_prop_1',
 'pre_util_prop_3',
 'enc_loans_credit_type_prop_0',
 'pre_till_pclose_prop_10',
 'pre_util_prop_6',
 'pre_loans_outstanding_prop_1',
 'pre_loans_credit_limit_prop_2',
 'pre_loans_credit_cost_rate_prop_6',
 'pre_loans_outstanding_prop_5',
 'pre_loans_credit_cost_rate_prop_11',
 'pre_loans_credit_cost_rate_prop_4',
 'pre_loans_next_pay_summ_prop_5',
 'pre_since_opened_prop_12',
 'pre_loans_credit_limit_prop_15',
 'enc_loans_credit_type_prop_2',
 'pre_fterm_prop_7',
 'enc_paym_0_prop_1',
 'is_zero_over2limit_prop_1',
 'pre_since_opened_prop_8',
 'pre_loans_max_overdue_sum_prop_1',
 'pre_loans_next_pay_summ_prop_0',
 'pre_pterm_prop_6',
 'pre_since_opened_prop_19',
 'is_zero_loans5_prop_1',
 'enc_loans_account_holder_type_prop_4',
 'pre_loans_credit_limit_prop_18',
 'pre_till_fclose_prop_4',
 'pre_pterm_prop_3',
 'is_zero_loans530_prop_1',
 'enc_loans_credit_status_prop_5',
 'pre_since_confirmed_prop_4',
 'pre_fterm_prop_3',
 'pre_till_fclose_prop_3',
 'pre_till_fcl

In [33]:
# Создадим список признаков исходного датасета из которых были сделаны пропорциональные фичи
prop_features_source_list = list(
    set(
        [
            re.sub(r'_prop.*$', '', col)
            for col in prop_features_result_list
        ]
    )
)

print(len(prop_features_source_list))
prop_features_source_list

22


['enc_loans_credit_status',
 'pre_loans_max_overdue_sum',
 'pre_loans_outstanding',
 'pre_since_opened',
 'enc_paym_0',
 'enc_paym_24',
 'pre_util',
 'pre_till_fclose',
 'enc_loans_account_holder_type',
 'is_zero_loans530',
 'is_zero_sum',
 'enc_loans_credit_type',
 'pre_loans_credit_limit',
 'pre_loans_next_pay_summ',
 'pre_pterm',
 'pre_since_confirmed',
 'pre_over2limit',
 'pre_fterm',
 'pre_till_pclose',
 'pre_loans_credit_cost_rate',
 'is_zero_over2limit',
 'is_zero_loans5']

In [34]:
# Соберем часть словаря пропорциональных фичей для пайплайна
prop_features_dict = {}

for source_col in prop_features_source_list:
    # Инициализируем пустой список для каждого исходного признака
    prop_features_dict[source_col] = []
    # Создадим паттерн: имя col в начале и после него подчёркивание или конец строки
    pattern = re.compile(r'^' + source_col + r'(_|$)')
    for result_col in prop_features_result_list:
        # Проверяем, совпадает ли имя признака с паттерном
        if pattern.match(result_col):
            # Ищем число в конце строки
            match = re.search(r'(\d+)$', result_col)
            # Добавляем найденное число в список для данного source_col
            prop_features_dict[source_col].append(int(match.group(1)))
prop_features_dict

{'enc_loans_credit_status': [5],
 'pre_loans_max_overdue_sum': [1],
 'pre_loans_outstanding': [1, 5],
 'pre_since_opened': [12, 8, 19],
 'enc_paym_0': [1],
 'enc_paym_24': [1],
 'pre_util': [3, 6],
 'pre_till_fclose': [4, 3, 1],
 'enc_loans_account_holder_type': [4],
 'is_zero_loans530': [1],
 'is_zero_sum': [1],
 'enc_loans_credit_type': [0, 2],
 'pre_loans_credit_limit': [2, 15, 18],
 'pre_loans_next_pay_summ': [5, 0],
 'pre_pterm': [6, 3],
 'pre_since_confirmed': [4, 7],
 'pre_over2limit': [17],
 'pre_fterm': [7, 3],
 'pre_till_pclose': [10, 7],
 'pre_loans_credit_cost_rate': [6, 11, 4],
 'is_zero_over2limit': [1],
 'is_zero_loans5': [1]}

In [35]:
"""
Добавим в словарь недостающие is_zero_loans* для функции суммирования.
Удалим is_zero_sum, фича is_zero_sum_prop_1 будет собираться другой функцией.
"""
is_zero_loans_list = [
        'is_zero_loans5',
        'is_zero_loans530',
        'is_zero_loans3060',
        'is_zero_loans6090',
        'is_zero_loans90'
    ]
for col in is_zero_loans_list:
    if col not in prop_features_dict.keys():
        prop_features_dict[col] = [1]
        
del prop_features_dict['is_zero_sum']

prop_features_dict

{'enc_loans_credit_status': [5],
 'pre_loans_max_overdue_sum': [1],
 'pre_loans_outstanding': [1, 5],
 'pre_since_opened': [12, 8, 19],
 'enc_paym_0': [1],
 'enc_paym_24': [1],
 'pre_util': [3, 6],
 'pre_till_fclose': [4, 3, 1],
 'enc_loans_account_holder_type': [4],
 'is_zero_loans530': [1],
 'enc_loans_credit_type': [0, 2],
 'pre_loans_credit_limit': [2, 15, 18],
 'pre_loans_next_pay_summ': [5, 0],
 'pre_pterm': [6, 3],
 'pre_since_confirmed': [4, 7],
 'pre_over2limit': [17],
 'pre_fterm': [7, 3],
 'pre_till_pclose': [10, 7],
 'pre_loans_credit_cost_rate': [6, 11, 4],
 'is_zero_over2limit': [1],
 'is_zero_loans5': [1],
 'is_zero_loans3060': [1],
 'is_zero_loans6090': [1],
 'is_zero_loans90': [1]}

# List for mean_value_frequency_feature_pipeline function

In [36]:
"""
Соберем список всех фичей средней частотности в итоговом датасете
созданных функцией create_mean_values_frequency_feature.
"""
mean_freq_result_list = [col for col in df_result_columns if 'mean_freq' in col]

print(len(mean_freq_result_list))
mean_freq_result_list

16


['pre_util_mean_freq',
 'pre_loans_credit_limit_mean_freq',
 'pre_since_opened_mean_freq',
 'pre_loans_credit_cost_rate_mean_freq',
 'enc_loans_credit_type_mean_freq',
 'pre_loans_next_pay_summ_mean_freq',
 'pre_since_confirmed_mean_freq',
 'pre_pterm_mean_freq',
 'enc_paym_0_mean_freq',
 'enc_loans_account_holder_type_mean_freq',
 'pre_loans530_mean_freq',
 'enc_paym_8_mean_freq',
 'pre_loans5_mean_freq',
 'enc_paym_10_mean_freq',
 'enc_loans_account_cur_mean_freq',
 'enc_paym_9_mean_freq']

In [37]:
"""
Соберем список признаков исходного датасета 
из которых были сделаны фичи средней частотности.
"""
mean_freq_source_list = [x[:-len('_mean_freq')] for x in mean_freq_result_list]
print(len(mean_freq_source_list))
mean_freq_source_list

16


['pre_util',
 'pre_loans_credit_limit',
 'pre_since_opened',
 'pre_loans_credit_cost_rate',
 'enc_loans_credit_type',
 'pre_loans_next_pay_summ',
 'pre_since_confirmed',
 'pre_pterm',
 'enc_paym_0',
 'enc_loans_account_holder_type',
 'pre_loans530',
 'enc_paym_8',
 'pre_loans5',
 'enc_paym_10',
 'enc_loans_account_cur',
 'enc_paym_9']

# Drop list

In [38]:
"""
Список временных фичей создаваемых
в процессы feature engineering и используемых для 
создания финальных фичей.
"""
temporary_features_list = [
    'enc_paym_avg_1_all',
    'enc_paym_avg_2_all',
    'enc_paym_avg_0_this_year',
    'enc_paym_avg_1_this_year',
    'enc_paym_avg_0_last_year',
    'is_zero_loans3060_prop_1',
    'is_zero_loans6090_prop_1',
    'is_zero_loans90_prop_1'
]

In [39]:
"""
Список признокав удаляемых из исходного датасета
функцикей drop_columns_drop_duplicates_pipeline
"""
drop_list = pre_features + temporary_features_list
print(len(drop_list))
drop_list

60


['id',
 'rn',
 'pre_since_opened',
 'pre_since_confirmed',
 'pre_pterm',
 'pre_fterm',
 'pre_till_pclose',
 'pre_till_fclose',
 'pre_loans_credit_limit',
 'pre_loans_next_pay_summ',
 'pre_loans_outstanding',
 'pre_loans_max_overdue_sum',
 'pre_loans_credit_cost_rate',
 'pre_loans5',
 'pre_loans530',
 'is_zero_loans5',
 'is_zero_loans530',
 'pre_util',
 'pre_over2limit',
 'is_zero_over2limit',
 'enc_paym_0',
 'enc_paym_1',
 'enc_paym_2',
 'enc_paym_8',
 'enc_paym_9',
 'enc_paym_10',
 'enc_paym_24',
 'enc_loans_account_holder_type',
 'enc_loans_credit_status',
 'enc_loans_credit_type',
 'enc_loans_account_cur',
 'is_zero_loans3060',
 'is_zero_loans6090',
 'is_zero_loans90',
 'enc_paym_3',
 'enc_paym_4',
 'enc_paym_5',
 'enc_paym_6',
 'enc_paym_7',
 'enc_paym_11',
 'enc_paym_12',
 'enc_paym_13',
 'enc_paym_14',
 'enc_paym_15',
 'enc_paym_16',
 'enc_paym_17',
 'enc_paym_18',
 'enc_paym_19',
 'enc_paym_20',
 'enc_paym_21',
 'enc_paym_22',
 'enc_paym_23',
 'enc_paym_avg_1_all',
 'enc_paym_av

# Hyperparameters and weights for ensemble models

In [43]:
# Соберём список словарей с гиперпараметрами моделей
params_list = []
for i in range(5):
    model = CatBoostClassifier()
    model.load_model(f'../models/fold_{i}_сonservative_model.bin')
    params_list.append(model.get_params())

with open('../models/best_сonservative_params.pkl', 'rb') as file:
    final_model_params = pickle.load(file)
    
params_list.append(final_model_params['best_all_folds_params'])
params_list

[{'verbose': 0,
  'use_best_model': True,
  'random_seed': 0,
  'grow_policy': 'SymmetricTree',
  'border_count': 113,
  'min_data_in_leaf': 5,
  'random_strength': 8.209932299,
  'learning_rate': 0.03625476076,
  'iterations': 3000,
  'l2_leaf_reg': 8.005778243,
  'boosting_type': 'Plain',
  'od_wait': 100,
  'depth': 4,
  'subsample': 0.9882297325,
  'bagging_temperature': 0.09710127579,
  'rsm': 0.7343256008,
  'eval_metric': 'AUC',
  'loss_function': 'Logloss',
  'auto_class_weights': 'Balanced'},
 {'verbose': 0,
  'use_best_model': True,
  'random_seed': 0,
  'grow_policy': 'SymmetricTree',
  'border_count': 113,
  'min_data_in_leaf': 5,
  'random_strength': 8.209932299,
  'learning_rate': 0.03625476076,
  'iterations': 3000,
  'l2_leaf_reg': 8.005778243,
  'boosting_type': 'Plain',
  'od_wait': 100,
  'depth': 4,
  'subsample': 0.9882297325,
  'bagging_temperature': 0.09710127579,
  'rsm': 0.7343256008,
  'eval_metric': 'AUC',
  'loss_function': 'Logloss',
  'auto_class_weights':

In [44]:
"""
Собираем список весов моделей,
для взвешивания финального предсказания ансамбля.
Для моделей фолдов это их AUC на валидационных наборах,
для финальной модели обученной на всех данных это средний 
AUC моделей фолдов.
"""
with open('../models/сonservative_models_weights.pkl', 'rb') as file:
    weights_list = pickle.load(file)

weights_list = weights_list['val_auc_list']
weights_list

[0.7576036850511159,
 0.7554545982995526,
 0.7532810994057619,
 0.7546988571803108,
 0.7524269260453276,
 0.7546930331964138]